In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp1_storage")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cbc': 'Version: 2.10.3', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
df = pd.DataFrame(result["data"]["rows"])
display(df.pivot_table(index=["n_people", "hops"], columns="backend",
                       values="median_ms").round(3))

backend        memory   neo4j  sqlite
n_people hops                        
50       1      0.036   0.711   0.091
         2      0.054   1.184   0.105
         3      0.075   1.986   0.284
         4      0.094   2.705   0.583
100      1      0.037   0.615   0.059
         2      0.057   1.320   0.135
         3      0.085   2.852   0.395
         4      0.111   5.296   0.962
200      1      0.041   0.646   0.084
         2      0.064   1.423   0.174
         3      0.096   4.085   0.538
         4      0.139   9.040   1.561
300      1      0.043   0.698   0.122
         2      0.071   1.716   0.253
         3      0.115   4.973   0.768
         4      0.172  13.045   2.268
500      1      0.050   0.575   0.175
         2      0.077   1.390   0.268
         3      0.114   5.002   0.629
         4      0.184  10.688   1.912
1000     1      0.068   0.727   0.352
         2      0.099   1.598   0.458
         3      0.160   6.515   0.978
         4      0.263  16.237   2.997

In [3]:
plots.exp1_crossover(result)   # figures/exp1_crossover.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp1_crossover.png')

## 관찰 요약

- 측정 범위(인원 50~1000, hops 1~4) 전 구간에서 **memory < sqlite < neo4j** 순서가 그대로 유지된다. "hops가 깊어지면 neo4j가 역전한다"는 가설은 이 범위에서 성립하지 않았다.
- hops=1 지연의 70~89%는 순수 IPC/프로토콜 바닥값(`protocol_floor.median_ms` ≈0.511ms)이 차지한다 — 얕은 hop 비교는 그래프 순회 비용보다는 bolt 왕복 비용을 주로 재는 셈이라, hops=1만 보고 백엔드를 비교하는 것은 오해를 부른다.
- neo4j/sqlite 배율은 hops=4 기준 4.6~5.8배 사이에서 평평하며(n=50: 4.64×, n=1000: 5.42×), 규모가 커진다고 좁혀지는 추세는 관측되지 않았다.
- `data.parity`가 전 조합에서 `match: true`이므로 위 지연 차이는 결과 집합 크기 차이가 아니라 백엔드 자체의 지연 특성이다. `data.skipped`는 비어 있다(neo4j 측정 전 구간 성공).
